# ⚡ Zalo AI Challenge 2026: DataMate — Đánh Giá Penalized Accuracy (PA)
## Đánh Giá Siêu Tốc Bằng vLLM Trên Tập Dữ Liệu Unseen (Không Trùng SFT)

> **Các điểm nâng cấp chính:**
> 1. **Tốc độ vLLM cực nhanh:** Thay vì mất 35 phút (như transformers tuần tự), vLLM hoàn thành toàn bộ 30-50 câu chỉ trong **~1 phút** trên GPU Colab!
> 2. **Chống Data Leakage (Né 890 câu SFT):** Tự động lọc sạch 100% các câu hỏi đã xuất hiện trong tập train `verified_sft_dataset.jsonl`, đảm bảo đánh giá khách quan trên các câu hỏi mô hình chưa từng thấy.
> 3. **Phân bố câu hỏi chuẩn bài thi:** Bao phủ đủ 8 kiểu đáp án: `number`, `percent`, `date`, `string`, `list`, `table`, `boolean`, `unanswerable` trên 24 doanh nghiệp.

### Bước 1: Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')

assert torch.cuda.is_available(), '❌ CUDA không khả dụng! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU hoặc A100.'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'\n🔥 Đã kết nối GPU thành công: {gpu_name} ({gpu_vram:.2f} GB VRAM)')

### Bước 2: Thiết lập Thư mục Dự án & Cài đặt vLLM

In [ ]:
import os, sys
from pathlib import Path

# Chuyển đến thư mục dự án trên Google Drive
PROJECT_PATH = Path('/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026')
assert PROJECT_PATH.exists(), f'❌ Không tìm thấy thư mục dự án tại: {PROJECT_PATH}'
os.chdir(str(PROJECT_PATH))
print(f'✅ Thư mục làm việc hiện tại: {os.getcwd()}')

# Cài đặt vLLM và các engine xử lý dữ liệu
!pip install -q "vllm>=0.7.0" python-calamine openpyxl xlrd duckdb tqdm peft transformers

### Bước 3: Kiểm tra Base Model Qwen2.5-Coder-7B-Instruct
Tải trực tiếp vào ổ cứng SSD cục bộ của Colab (`/content/models`) nếu chưa có.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

base_model_dir = Path("/content/models/Qwen2.5-Coder-7B-Instruct")
if not (base_model_dir / "config.json").exists():
    print("🚀 Đang tải Base Model Qwen2.5-Coder-7B-Instruct từ Hugging Face về Colab...")
    snapshot_download(
        repo_id="Qwen/Qwen2.5-Coder-7B-Instruct",
        local_dir=str(base_model_dir),
        local_dir_use_symlinks=False,
        resume_download=True,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
    print(f"\n✅ Tải thành công Base Model tại: {base_model_dir}")
else:
    print(f"✅ Base Model đã có sẵn tại: {base_model_dir}")

### Bước 4: Đánh Giá Bằng vLLM Trên 30 Câu Hỏi Unseen (0% Trùng SFT)
- **Thời gian chạy**: ~45 – 60 giây (nhanh hơn 40 lần so với trước).
- **Dữ liệu**: 30 câu hỏi mới tinh, chưa từng xuất hiện trong tập huấn luyện SFT.

In [ ]:
# Chạy đánh giá vLLM trên 30 câu Unseen sạch (0% trùng SFT)
!python scripts/evaluate_pilot_30.py \
    --backend vllm \
    --dtype float16 \
    --gpu-memory-utilization 0.90 \
    --adapter-path "Model/V2" \
    --base-model "/content/models/Qwen2.5-Coder-7B-Instruct" \
    --questions-file "result/unseen_eval_30/questions.jsonl" \
    --gold-file "eval_gold/unseen_gold_30.jsonl" \
    --data-dir "data/train" \
    --output-dir "result/eval_qwen7b_unseen30"


### Bước 5 (Tùy chọn): Đánh Giá Mở Rộng Trên 50 Câu Hỏi Unseen
Nếu muốn kiểm chứng độ ổn định và độ tổng quát hóa cao hơn nữa.

In [ ]:
# (Tùy chọn) Chạy đánh giá vLLM trên 50 câu Unseen sạch
!python scripts/evaluate_pilot_30.py \
    --backend vllm \
    --dtype float16 \
    --gpu-memory-utilization 0.90 \
    --adapter-path "Model/V2" \
    --base-model "/content/models/Qwen2.5-Coder-7B-Instruct" \
    --questions-file "result/unseen_eval_50/questions.jsonl" \
    --gold-file "eval_gold/unseen_gold_50.jsonl" \
    --data-dir "data/train" \
    --output-dir "result/eval_qwen7b_unseen50"
